# PAUL Open Model — First Model Validation (Gemma 4 E4B IT)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/foundrypaul-cloud/paul-open/blob/main/notebooks/02_first_model_validation_e4b.ipynb)

This notebook executes the **First Model Validation Experiment** on Google Colab (Tesla T4, 14.56 GiB usable VRAM).

### Experiment Scope & Constraints
- **Target Model**: `google/gemma-4-E4B-it` (~4.5B dense parameters, edge-optimized multimodal).
- **Architecture**: `AutoModelForMultimodalLM` (official Gemma 4 Transformers API).
- **Quantization**: 4-bit NF4 QLoRA-compatible configuration via `BitsAndBytesConfig` (`dtype=torch.float16`).
- **Hardware Runtime**: Tesla T4 (14.56 GiB usable VRAM, Turing CC 7.5).
- **Safety & Scope Boundaries**:
  - Baseline validation only (zero dataset creation, zero training, zero weights uploaded).
  - Safe for one-click **"Run all"** execution in Google Colab.
  - Complete GPU memory cleanup and peak VRAM audit.

## Step 1: Repository Setup & Explicit Package Installation
Clone the repository (if running in Colab) and pin verified, compatible versions of all core dependencies before importing them.

In [ ]:
import os
import sys

# In Google Colab, clone the repository to access local configs and src package
if "google.colab" in sys.modules or os.environ.get("COLAB_GPU") is not None:
    if not os.path.exists("paul-open"):
        !git clone -q https://github.com/foundrypaul-cloud/paul-open.git
        %cd paul-open
    elif os.path.basename(os.getcwd()) != "paul-open":
        %cd paul-open

# Install explicit, pinned versions of the Gemma 4 ML stack
# Google's Gemma 4 documentation requires Transformers >=5.10.1 and PEFT >=0.19.0
!pip install -q --no-cache-dir \
    "torch>=2.11.0" \
    "transformers>=5.13.1" \
    "peft>=0.19.0" \
    "bitsandbytes>=0.45.0" \
    "accelerate>=1.2.0" \
    "datasets>=3.2.0" \
    "huggingface_hub>=0.28.0" \
    "pyyaml>=6.0" \
    "rich>=13.0.0" \
    "sentencepiece>=0.2.0" \
    "tokenizers>=0.21.0"

## Step 2: Hardware Preflight & Dependency Audit
Verify GPU device capabilities, VRAM availability, CUDA version, and installed dependency versions.

In [ ]:
import platform
import accelerate
import bitsandbytes as bnb
import huggingface_hub
import peft
import torch
import transformers

print("=" * 70)
print(" HARDWARE PREFLIGHT & DEPENDENCY AUDIT")
print("=" * 70)
print(f" Python Version      : {platform.python_version()}")
print(f" OS / Platform       : {platform.platform()}")
print(f" PyTorch Version     : {torch.__version__}")
print(f" Transformers Version: {transformers.__version__} (>=5.10.1 required for Gemma 4)")
print(f" PEFT Version        : {peft.__version__} (>=0.19.0 required for Gemma 4 QLoRA)")
print(f" BitsAndBytes Version: {bnb.__version__} (>=0.45.0 required)")
print(f" Accelerate Version  : {accelerate.__version__}")
print(f" Hugging Face Hub    : {huggingface_hub.__version__}")
print("-" * 70)

if not torch.cuda.is_available():
    raise SystemError(
        "CRITICAL: No CUDA GPU detected! Please select: "
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then rerun."
    )

gpu_props = torch.cuda.get_device_properties(0)
gpu_name = gpu_props.name
total_vram_gb = gpu_props.total_memory / (1024 ** 3)
usable_vram_gb = total_vram_gb * 0.95
cuda_version = torch.version.cuda
cc = f"{gpu_props.major}.{gpu_props.minor}"

print(f" GPU Device          : {gpu_name}")
print(f" Compute Capability  : {cc}")
print(f" Total VRAM          : {total_vram_gb:.2f} GiB")
print(f" Usable VRAM Est.    : {usable_vram_gb:.2f} GiB")
print(f" CUDA Runtime        : {cuda_version}")

if total_vram_gb < 6.0:
    raise SystemError(
        f"CRITICAL: Insufficient VRAM ({total_vram_gb:.2f} GiB). Gemma 4 E4B in 4-bit requires >=6.0 GiB VRAM."
    )

print("✓ Hardware preflight checks PASSED.")
print("=" * 70)

## Step 3: Secure Hugging Face Authentication & Access Verification
Retrieve `HF_TOKEN` from Google Colab Secrets without exposing the secret in output or files, and verify access to `google/gemma-4-E4B-it`.

In [ ]:
import os
from huggingface_hub import HfApi, login

MODEL_ID = "google/gemma-4-E4B-it"

# Securely fetch token from Colab Secrets (or fallback to environment variable)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "MISSING HF_TOKEN: Please configure your Hugging Face token in Google Colab Secrets:\n"
        "1. Click the Key icon (Secrets) on the left sidebar.\n"
        "2. Add secret with Name: 'HF_TOKEN' and your token as the Value.\n"
        "3. Toggle 'Notebook access' ON for this secret.\n"
        "4. Rerun this notebook."
    )

# Authenticate without printing token or storing to git credentials
login(token=hf_token, add_to_git_credential=False)
print("✓ Hugging Face authenticated securely.")

# Verify repository access permissions
api = HfApi()
try:
    model_info = api.model_info(MODEL_ID, token=hf_token)
    print(f"✓ Access verified for gated model: {MODEL_ID}")
    print(f"  - Pipeline tag : {model_info.pipeline_tag}")
    print(f"  - Model Sha    : {model_info.sha[:12]}")
except Exception as e:
    raise PermissionError(
        f"ACCESS DENIED to '{MODEL_ID}'.\n"
        f"Please ensure you have accepted Google's Gemma license agreement at:\n"
        f"https://huggingface.co/{MODEL_ID}\n"
        f"Then verify that your HF_TOKEN has read access to this model."
    ) from e

## Step 4: 4-Bit QLoRA Quantization & Model Loading
Configure `BitsAndBytesConfig` (4-bit NF4 with double quantization and float16 compute dtype) and load `AutoProcessor` and `AutoModelForMultimodalLM`.

In [ ]:
import gc
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig

def get_vram_stats():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        max_alloc = torch.cuda.max_memory_allocated() / (1024 ** 3)
        max_res = torch.cuda.max_memory_reserved() / (1024 ** 3)
        return {
            "allocated_gb": allocated,
            "reserved_gb": reserved,
            "max_allocated_gb": max_alloc,
            "max_reserved_gb": max_res,
        }
    return {"allocated_gb": 0.0, "reserved_gb": 0.0, "max_allocated_gb": 0.0, "max_reserved_gb": 0.0}

vram_baseline = get_vram_stats()
print(f"[VRAM Baseline] Allocated: {vram_baseline['allocated_gb']:.2f} GiB | Reserved: {vram_baseline['reserved_gb']:.2f} GiB")

# Configure 4-bit NF4 QLoRA quantization (float16 compute dtype for Tesla T4 Turing architecture)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Using compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)

print(f"Loading {MODEL_ID} weights via AutoModelForMultimodalLM in 4-bit NF4...")
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    token=hf_token,
)

vram_post_load = get_vram_stats()
print(f"[VRAM Post-Load] Allocated: {vram_post_load['allocated_gb']:.2f} GiB | Reserved: {vram_post_load['reserved_gb']:.2f} GiB")
print(f"✓ Model weights loaded successfully (Weight Memory Footprint: ~{vram_post_load['allocated_gb'] - vram_baseline['allocated_gb']:.2f} GiB).")

## Step 5: Chat Template Formatting & Baseline Inference Execution
Execute the single baseline science education prompt using Gemma 4's native chat template with `enable_thinking=False`.

In [ ]:
import time

# Baseline science tutoring test prompt
PROMPT_TEXT = "Explain photosynthesis to a Grade 8 student in a clear, empathetic and scientifically accurate way."
messages = [
    {
        "role": "user",
        "content": PROMPT_TEXT,
    }
]

# Apply native Gemma 4 chat template
formatted_prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print("--- Formatted Input Prompt ---")
print(formatted_prompt)
print("------------------------------")

inputs = processor(text=formatted_prompt, return_tensors="pt").to(model.device)
vram_post_tokenize = get_vram_stats()
print(f"[VRAM Post-Tokenize] Allocated: {vram_post_tokenize['allocated_gb']:.2f} GiB")

# Execute generation with latency timing
start_time = time.perf_counter()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
latency_seconds = time.perf_counter() - start_time

# Extract generated tokens
input_len = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_len:]
generated_response = processor.decode(generated_tokens, skip_special_tokens=True).strip()

vram_post_gen = get_vram_stats()

print("\n" + "=" * 70)
print(" MODEL GENERATION RESPONSE")
print("=" * 70)
print(generated_response)
print("=" * 70)
print(f" Generation Latency  : {latency_seconds:.3f} s ({len(generated_tokens)} tokens, {len(generated_tokens) / latency_seconds:.1f} tok/s)")
print(f" [VRAM Post-Gen]     : Allocated {vram_post_gen['allocated_gb']:.2f} GiB | Peak {vram_post_gen['max_allocated_gb']:.2f} GiB")

## Step 6: GPU Memory Cleanup & Machine-Readable Validation Summary
Release all GPU tensors, clear CUDA cache, and produce the structured validation report.

In [ ]:
# Record final peak metrics before tensor deletion
peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
peak_reserved_gb = torch.cuda.max_memory_reserved() / (1024 ** 3) if torch.cuda.is_available() else 0.0

# Comprehensive GPU cleanup
del model
del processor
del inputs
del outputs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

vram_after_cleanup = get_vram_stats()

# Construct machine-readable summary
validation_summary = {
    "validation_status": "PASSED",
    "model_id": MODEL_ID,
    "gpu": gpu_name if 'gpu_name' in locals() else "N/A",
    "vram_gb": round(total_vram_gb, 2) if 'total_vram_gb' in locals() else 0.0,
    "quantization": "4-bit NF4 (Double Quantization)",
    "peak_vram_gb": round(peak_vram_gb, 2),
    "peak_reserved_gb": round(peak_reserved_gb, 2),
    "inference_success": len(generated_response) > 0,
    "generation_latency_seconds": round(latency_seconds, 3) if 'latency_seconds' in locals() else 0.0,
    "vram_retained_after_cleanup_gb": round(vram_after_cleanup['allocated_gb'], 2),
}

print("=" * 70)
print(" FIRST MODEL VALIDATION SUMMARY (Tesla T4 Free Tier)")
print("=" * 70)
for k, v in validation_summary.items():
    print(f" {k:<32}: {v}")
print("=" * 70)
print("✓ VALIDATION COMPLETED SUCCESSFULLY. Ready for research progression.")